# RNN for repeated ones

A recurrent network reads a sequence one element at a time and carries a hidden state between steps. This notebook shows what that state can store: the task is to output 1 whenever the current and the previous inputs are both 1. We first set the weights by hand, then train the same kind of network from data.

The hand-crafted network is adapted from the [code of CS231n Lecture 7](https://colab.research.google.com/drive/15YMP5dVcfUz2M36a_wmL4mEqKUlgJrdE?usp=sharing).

**Aprendizado Profundo para Reconhecimento Visual** — Ifes Campus Serra
Autor: Thiago M. Paixão

In [ ]:
# If True, test is running on Colab. Otherwise, test is assumed to be offline.
TEST_ON_COLAB = True
FOLDERNAME = 'Colab Notebooks/Aprendizado Profundo para Reconhecimento Visual/07_rnn'  # only used if TEST_ON_COLAB is True

assert not (FOLDERNAME is None and TEST_ON_COLAB), "FOLDERNAME has to be set if TEST_ON_COLAB is True"

In [ ]:
# Create dataset directory
import os

if TEST_ON_COLAB:
    # This mounts your Google Drive to the Colab VM.
    from google.colab import drive

    drive.mount('/content/drive')
    os.chdir(f'/content/drive/My Drive/{FOLDERNAME}')

## Hand-crafted RNN

The network follows the recurrence of a vanilla RNN, with ReLU in place of the usual tanh and no bias:

$$h_t = \text{ReLU}(W_{hh} h_{t-1} + W_{xh} x_t), \qquad y_t = \text{ReLU}(W_{yh} h_t)$$

To detect repeated ones, the hidden state only needs three values: $h_t = (x_t, x_{t-1}, 1)$. Each weight has a simple role:

- $W_{xh}$ writes the current input into the first position.
- The first row of $W_{hh}$ is zero, so the first position is overwritten at each step.
- The second row of $W_{hh}$ copies the first position into the second one, which keeps the previous input.
- The third row of $W_{hh}$ keeps the constant 1, which acts as a bias.
- $W_{yh}$ computes $x_t + x_{t-1} - 1$. After the ReLU, this is 1 only when both inputs are 1.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

h_0 = np.array([0, 0, 1])
W_xh = np.array([1, 0, 0])
W_hh = np.array([[0, 0, 0],
                 [1, 0, 0],
                 [0, 0, 1]])
W_yh = np.array([1, 1, -1])


def relu(x):
    return np.maximum(0, x)


def rnn_forward(x_seq):
    h = h_0
    hidden_states, outputs = [], []
    for x in x_seq:
        h = relu(W_hh @ h + W_xh * x)  # the same weights are used at every step
        hidden_states.append(h)
        outputs.append(relu(W_yh @ h))
    return np.array(hidden_states), np.array(outputs)


x_seq = [0, 1, 0, 1, 1, 1, 0, 1, 1, 0, 1, 1, 0, 1, 0, 0, 0, 0, 1, 1]
hidden_states, outputs = rnn_forward(x_seq)
print("Inputs: ", x_seq)
print("Outputs:", outputs.tolist())

The plot shows the hidden state at each step. The second row is the first row shifted by one step, and the output is 1 where both rows are 1.

In [ ]:
plt.figure(figsize=(10, 2))
plt.imshow(hidden_states.T, cmap="viridis")
plt.xticks(range(len(x_seq)), x_seq)
plt.yticks(range(3), ["$x_t$", "$x_{t-1}$", "constant"])
plt.xlabel("Input sequence")
plt.title("Hand-crafted hidden state")
plt.show()

## Learning the weights

In practice, nobody sets these weights by hand. Now the network only sees pairs of input and target sequences, and backpropagation has to find a way of remembering the previous input.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(42)


def make_batch(batch_size, seq_len):
    x = torch.randint(0, 2, (batch_size, seq_len, 1), device=device).float()
    previous = F.pad(x[:, :-1], (0, 0, 1, 0))  # the input of the previous step, 0 at the first step
    return x, x * previous


class RepeatedOnes(nn.Module):
    def __init__(self, hidden_size=4):
        super().__init__()
        self.rnn = nn.RNN(1, hidden_size, batch_first=True)
        self.output = nn.Linear(hidden_size, 1)

    def forward(self, x):
        states, _ = self.rnn(x)  # states: all hidden states, of shape (N, T, H)
        return self.output(states), states  # one logit per step: (N, T, 1)


model = RepeatedOnes().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)
loss_history = []
for step in range(500):
    x, y = make_batch(batch_size=64, seq_len=20)
    logits, _ = model(x)
    loss = F.binary_cross_entropy_with_logits(logits, y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    loss_history.append(loss.item())

plt.plot(loss_history)
plt.xlabel("Step")
plt.ylabel("Loss")
plt.title("Loss History")
plt.show()

The weights do not depend on the position in the sequence, so the trained network also works on sequences much longer than the ones seen during training.

In [ ]:
with torch.no_grad():
    x, y = make_batch(batch_size=1000, seq_len=200)
    logits, _ = model(x)
    accuracy = ((logits > 0).float() == y).float().mean().item()
print(f"Accuracy on sequences of length 200: {accuracy:.4f}")

with torch.no_grad():
    x = torch.tensor(x_seq, device=device).float().reshape(1, -1, 1)
    logits, states = model(x)
print("Inputs: ", x_seq)
print("Outputs:", (logits[0, :, 0] > 0).int().tolist())

The learned hidden state solves the same task, but it does not need to look like the hand-crafted one: its values are not simply $x_t$ and $x_{t-1}$, and they are harder to interpret.

In [ ]:
plt.figure(figsize=(10, 2))
plt.imshow(states[0].T.cpu(), cmap="viridis")
plt.xticks(range(len(x_seq)), x_seq)
plt.yticks(range(states.size(2)))
plt.xlabel("Input sequence")
plt.ylabel("Hidden unit")
plt.title("Learned hidden state")
plt.colorbar()
plt.show()